# Treino completo ABSA QLoRA

T4 x2 no Kaggle com o worker restrito à GPU 0. Saldo mínimo de 1 h e limite total de sessão de 60 min. Reserve tempo para exportar e encerrar a sessão. A configuração reproduz `configs/absa-qlora-v1.json` e os dados vêm do commit `82880bcff61d9c32cd910460a05eee6ee78cd858`.

O treino usa 1.263 exemplos e 140 de validação. O checkpoint escolhido minimiza `eval_loss`. Gold humano e templates reservados ficam para a avaliação final.

In [ ]:
import json
from pathlib import Path
from uuid import uuid4

GPU_HOURS_REMAINING = None
WORKER_LIMIT_MINUTES = 50
if type(GPU_HOURS_REMAINING) not in (float, int) or GPU_HOURS_REMAINING < 1:
    raise ValueError("Informe o saldo real de GPU em horas. O treino exige 1 h disponível.")
if not 1 <= WORKER_LIMIT_MINUTES <= 50:
    raise ValueError("Limite do worker fora do orçamento da sessão.")
RUN = Path("/kaggle/working") / ("absa-qlora-train-" + uuid4().hex[:8])
RUN.mkdir(exist_ok=False)
CONFIG_PATH = RUN / "config.json"
CONFIG_PATH.write_text(
    """{
  "run_version": "absa-qlora-v1",
  "model": {
    "id": "Qwen/Qwen2.5-1.5B-Instruct",
    "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    "attn_implementation": "sdpa"
  },
  "data": {
    "manifest": "data/absa/dataset-v1.json",
    "manifest_sha256_lf": "0189e2b5f6018e68c80d3ff9332110e63b5b6a527a79535a98074bda2e4470a5",
    "train": "data/absa/train-v1.jsonl",
    "validation": "data/absa/validation-v1.jsonl"
  },
  "runtime": {
    "accelerator": "Tesla T4",
    "cuda": "13.0",
    "visible_devices": "0",
    "packages": {
      "torch": "2.11.0+cu130",
      "transformers": "5.20.2",
      "bitsandbytes": "0.50.2",
      "accelerate": "1.15.0",
      "trl": "1.17.1",
      "peft": "0.21.2",
      "datasets": "5.1.0"
    },
    "trainable_dtype_after_sft_init": "float32",
    "session_time_limit_minutes": 60
  },
  "quantization": {
    "load_in_4bit": true,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "bnb_4bit_compute_dtype": "float16"
  },
  "lora": {
    "r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "target_modules": "all-linear",
    "task_type": "CAUSAL_LM"
  },
  "trainer": {
    "num_train_epochs": 3,
    "max_length": 256,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 5,
    "completion_only_loss": true,
    "assistant_only_loss": false,
    "packing": false,
    "dataloader_drop_last": false,
    "learning_rate": 0.0002,
    "lr_scheduler_type": "constant",
    "warmup_steps": 0,
    "optim": "adamw_torch",
    "fp16": true,
    "bf16": false,
    "gradient_checkpointing": true,
    "gradient_checkpointing_kwargs": {
      "use_reentrant": false
    },
    "seed": 2718,
    "data_seed": 2718,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 3,
    "load_best_model_at_end": true,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": false,
    "logging_steps": 10,
    "logging_nan_inf_filter": false,
    "report_to": "none"
  }
}
""",
    encoding="utf-8",
)
print("Saída:", RUN, flush=True)

In [ ]:
%%writefile /kaggle/working/absa_qlora_train_worker.py
import shutil
import hashlib
import inspect
import json
import math
import os
import platform
import subprocess
import sys
import time
import traceback
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from statistics import median
from urllib.request import urlopen

ROOT = Path(sys.argv[1])
QUOTA = float(sys.argv[2])
CONFIG = json.loads(Path(sys.argv[3]).read_text(encoding="utf-8"))
BASE = CONFIG["model"]["id"]
REV = CONFIG["model"]["revision"]
SOURCE = "82880bcff61d9c32cd910460a05eee6ee78cd858"
RUNTIME = ("torch", "transformers", "bitsandbytes", "accelerate")
PACKAGES = (*CONFIG["runtime"]["packages"], "tokenizers", "huggingface-hub", "pyarrow", "dill")
SEED = CONFIG["trainer"]["seed"]
GIB = 2**30
started = time.perf_counter()
report = dict(status="running", stage="start", model=BASE, revision=REV, source_commit=SOURCE,
              weekly_gpu_hours_remaining_at_start=QUOTA, python=platform.python_version(),
              config=CONFIG, worker_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())


def versions(names):
    result = {}
    for name in names:
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = None
    return result


def record(stage, **values):
    report.update(stage=stage, elapsed_s=time.perf_counter() - started, **values)
    path = ROOT / "training.json"
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)
    print(f"stage={stage}", flush=True)


def main():
    runtime = versions(RUNTIME)
    record("preflight", packages_before=versions(PACKAGES))
    import torch

    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Select a T4 accelerator. The worker must see exactly one CUDA device.")
    gpu = torch.cuda.get_device_properties(0)
    record("gpu_identified", gpu=gpu.name, cuda=torch.version.cuda,
           gpu_capability=[gpu.major, gpu.minor], gpu_total_gib=gpu.total_memory / GIB)
    expected = CONFIG["runtime"]["packages"]
    if ("T4" not in gpu.name or torch.version.cuda != CONFIG["runtime"]["cuda"]
            or any(runtime[name] != expected[name] for name in RUNTIME if name != "bitsandbytes")
            or runtime["bitsandbytes"] not in (None, expected["bitsandbytes"])):
        raise RuntimeError("Kaggle image differs from the validated CUDA stack")
    constraints = ROOT / "constraints.txt"
    constraints.write_text("\n".join(f"{name}=={expected[name]}" for name in RUNTIME) + "\n")
    pins = [f"{name}=={expected[name]}" for name in ("trl", "peft", "datasets", "bitsandbytes")]
    record("install", install_pins=pins)
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    "--constraint", str(constraints), *pins], check=True)
    record("import", packages=versions(PACKAGES))
    if versions(expected) != expected:
        raise RuntimeError("Installed versions differ from the frozen run configuration")
    import bitsandbytes as bnb
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (
        AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback, set_seed,
    )
    from trl import SFTConfig, SFTTrainer

    if "processing_class" not in inspect.signature(SFTTrainer).parameters:
        raise RuntimeError("Unexpected installed SFTTrainer signature")
    required = {"max_length", "completion_only_loss", "assistant_only_loss"}
    if not required <= set(inspect.signature(SFTConfig).parameters):
        raise RuntimeError("Unexpected installed SFTConfig signature")
    set_seed(SEED)
    record("dataset")
    def read_verified(relative, checksum):
        url = f"https://raw.githubusercontent.com/JoaoNordemann/partner-copilot/{SOURCE}/{relative}"
        with urlopen(url, timeout=30) as response:
            raw = response.read().replace(b"\r\n", b"\n")
        if hashlib.sha256(raw).hexdigest() != checksum:
            raise ValueError(f"Dataset hash mismatch: {relative}")
        return raw

    data = CONFIG["data"]
    manifest = json.loads(read_verified(data["manifest"], data["manifest_sha256_lf"]))
    examples = {}
    identifiers = {}
    for name in ("train", "validation"):
        split = manifest["splits"][name]
        raw = read_verified(data[name], split["sha256_lf"])
        rows = list(map(json.loads, raw.decode("utf-8").splitlines()))
        identifiers[name] = {row["id"] for row in rows}
        if len(rows) != len(identifiers[name]) or len(rows) != manifest["split_counts"][name]:
            raise ValueError(f"Unexpected split count: {name}")
        examples[name] = []
        for row in rows:
            messages = row["messages"]
            if [message["role"] for message in messages] != ["user", "assistant"]:
                raise ValueError(f"Unexpected message roles: {row['id']}")
            examples[name].append({"prompt": messages[:1], "completion": messages[1:]})
    if identifiers["train"] & identifiers["validation"]:
        raise ValueError("Train and validation IDs overlap")
    datasets = {
        name: Dataset.from_generator(lambda: iter(rows), cache_dir=str(ROOT / f"cache-{name}"))
        for name, rows in examples.items()
    }
    record("model_load", split_counts={name: len(rows) for name, rows in examples.items()},
           split_hashes={name: manifest["splits"][name]["sha256_lf"] for name in examples})
    tokenizer = AutoTokenizer.from_pretrained(BASE, revision=REV, token=False)
    if hashlib.sha256(tokenizer.chat_template.encode()).hexdigest() != manifest["chat_template_sha256"]:
        raise ValueError("Tokenizer chat template differs from the dataset profile")
    config4 = BitsAndBytesConfig(**{**CONFIG["quantization"], "bnb_4bit_compute_dtype": torch.float16})
    started_load = time.perf_counter()

    free, total = torch.cuda.mem_get_info(0)
    torch.cuda.reset_peak_memory_stats(0)
    model = AutoModelForCausalLM.from_pretrained(
        BASE, revision=REV, token=False, device_map={"": 0}, quantization_config=config4,
        dtype=torch.float16, attn_implementation=CONFIG["model"]["attn_implementation"],
    )
    record("prepare_lora", model_load_s=time.perf_counter() - started_load)
    if not getattr(model, "is_loaded_in_4bit", False) or not any(
        isinstance(layer, bnb.nn.Linear4bit) for layer in model.modules()
    ):
        raise RuntimeError("Base model did not load with BNB 4-bit layers")
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(**CONFIG["lora"], revision=REV))
    trainable = {name: param.numel() for name, param in model.named_parameters() if param.requires_grad}
    if not trainable or any("lora_" not in name for name in trainable):
        raise RuntimeError("Expected only LoRA parameters to be trainable")
    record("sft_init", trainable_parameters=sum(trainable.values()))

    class StepTimer(TrainerCallback):
        def __init__(self):
            self.seconds = []

        def on_step_begin(self, args, state, control, **kwargs):
            torch.cuda.synchronize()
            self.step_start = time.perf_counter()

        def on_step_end(self, args, state, control, **kwargs):
            torch.cuda.synchronize()
            self.seconds.append(time.perf_counter() - self.step_start)
            if state.global_step % 10 == 0:
                record("train", completed_steps=state.global_step, step_seconds=self.seconds,
                       peak_reserved_gib=torch.cuda.max_memory_reserved(0) / GIB)
                print(f"steps={state.global_step} epoch={state.epoch:.3f}", flush=True)

        def on_save(self, args, state, control, **kwargs):
            record("checkpoint", completed_steps=state.global_step,
                   best_checkpoint=state.best_model_checkpoint, log_history=state.log_history)
            print(f"saved={state.global_step} epoch={state.epoch:.3f} best={state.best_metric}", flush=True)

    timer = StepTimer()
    args = SFTConfig(output_dir=str(ROOT / "trainer"), eos_token=tokenizer.eos_token,
                     **CONFIG["trainer"])
    trainer = SFTTrainer(model=model, args=args, processing_class=tokenizer,
                         train_dataset=datasets["train"], eval_dataset=datasets["validation"],
                         callbacks=[timer])
    # TRL 1.17.1 casts trainable quantized adapters to BF16 at SFTTrainer init.
    # T4 FP16 GradScaler needs supported gradient dtypes. Keep LoRA masters FP32.
    adapters = [parameter for parameter in trainer.model.parameters() if parameter.requires_grad]
    initial_dtypes = sorted({str(parameter.dtype) for parameter in adapters})
    with torch.no_grad():
        for parameter in adapters:
            parameter.data = parameter.data.float()
    if not adapters or any(parameter.dtype != torch.float32 for parameter in adapters):
        raise RuntimeError("FP32 LoRA master weights were not applied after SFT initialization")
    record("completion_mask", adapter_dtypes_after_trl_init=initial_dtypes,
           trainable_adapter_dtype="torch.float32")
    for name, dataset in (("train", trainer.train_dataset), ("validation", trainer.eval_dataset)):
        if len(dataset) != len(examples[name]):
            raise ValueError(f"SFT preprocessing dropped rows from {name}")
        for i, example in enumerate(examples[name]):
            batch = trainer.data_collator([dataset[i]])
            ids = batch["input_ids"][0].tolist()
            labels = batch["labels"][0].tolist()
            prefix = tokenizer.apply_chat_template(example["prompt"], tokenize=True,
                                                   add_generation_prompt=True, return_dict=True)["input_ids"]
            boundary = len(prefix)
            if not (0 < boundary < len(ids) <= args.max_length and ids[:boundary] == prefix
                    and labels[:boundary] == [-100] * boundary and labels[boundary:] == ids[boundary:]):
                raise ValueError(f"Invalid completion mask at {name}/{i}")
            if tokenizer.decode(labels[boundary:], skip_special_tokens=True).strip() != example["completion"][0]["content"]:
                raise ValueError(f"Truncated or incorrect JSON at {name}/{i}")
    record("train", masks_verified={name: len(rows) for name, rows in examples.items()})
    train_start = time.perf_counter()
    trainer.train()
    history = trainer.state.log_history
    validations = [entry for entry in history if "eval_loss" in entry]
    training = [entry["loss"] for entry in history if "loss" in entry]
    steps = math.ceil(len(examples["train"]) / args.gradient_accumulation_steps) * int(args.num_train_epochs)
    if (trainer.state.global_step != steps or len(validations) != int(args.num_train_epochs)
            or not training or not all(map(math.isfinite, training))
            or not all(math.isfinite(entry["eval_loss"]) for entry in validations)):
        raise RuntimeError("Training did not finish with complete finite metrics")
    winner = min(validations, key=lambda entry: (entry["eval_loss"], entry["step"]))
    checkpoint = Path(trainer.state.best_model_checkpoint)
    if checkpoint.name != f"checkpoint-{winner['step']}" or not checkpoint.is_dir():
        raise RuntimeError("Trainer best checkpoint differs from minimum validation loss")
    adapter = ROOT / "adapter"
    adapter.mkdir()
    for name in ("adapter_config.json", "adapter_model.safetensors"):
        shutil.copyfile(checkpoint / name, adapter / name)
    tokenizer.save_pretrained(adapter)
    record("complete", status="passed", completed_steps=trainer.state.global_step,
           validation_history=validations, log_history=history, selected=winner,
           best_checkpoint=checkpoint.name, train_elapsed_s=time.perf_counter() - train_start,
           adapter_sha256=hashlib.sha256((adapter / "adapter_model.safetensors").read_bytes()).hexdigest(),
           peak_allocated_gib=torch.cuda.max_memory_allocated(0) / GIB,
           peak_reserved_gib=torch.cuda.max_memory_reserved(0) / GIB,
           estimated_free_headroom_gib=(free - torch.cuda.max_memory_reserved(0)) / GIB,
           median_seconds_per_optimizer_step=median(timer.seconds[2:]), step_seconds=timer.seconds)
    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except Exception as exc:
        record(report["stage"], status="error", packages=versions(PACKAGES),
               error_type=type(exc).__name__, error=str(exc), traceback=traceback.format_exc())
        raise


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

worker = Path("/kaggle/working/absa_qlora_train_worker.py")
log = RUN / "training.log"
with (
    log.open("w", encoding="utf-8") as stream,
    subprocess.Popen(
        [
            "timeout",
            "--signal=TERM",
            "--kill-after=5s",
            f"{WORKER_LIMIT_MINUTES}m",
            sys.executable,
            "-u",
            str(worker),
            str(RUN),
            str(GPU_HOURS_REMAINING),
            str(CONFIG_PATH),
        ],
        env={
            **os.environ,
            "CUDA_VISIBLE_DEVICES": "0",
            "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
            "TOKENIZERS_PARALLELISM": "false",
        },
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    ) as proc,
):
    for line in proc.stdout:
        stream.write(line)
        if line.startswith(("stage=", "steps=", "saved=")):
            print(line.rstrip(), flush=True)
    returncode = proc.wait()
report_file = RUN / "training.json"
report = json.loads(report_file.read_text(encoding="utf-8")) if report_file.exists() else {}
report["worker_exit_code"] = returncode
report["worker_limit_minutes"] = WORKER_LIMIT_MINUTES
if returncode in (124, 137):
    report.update(status="time_limit", error="Worker reached the timebox")
elif returncode and report.get("status") in (None, "running"):
    report.update(status="worker_error", error=f"Worker exited with code {returncode}")
report_file.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
bundle = RUN.parent / f"{RUN.name}.zip"
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    for artifact in (report_file, log, CONFIG_PATH, RUN / "constraints.txt"):
        if artifact.is_file():
            archive.write(artifact, arcname=artifact.name)
    for artifact in sorted((RUN / "adapter").rglob("*")):
        if artifact.is_file():
            archive.write(artifact, arcname=artifact.relative_to(RUN))
summary = {k: v for k, v in report.items() if k not in ("step_seconds", "log_history", "config")}
print("TRAINING_REPORT=" + json.dumps(summary, ensure_ascii=False), flush=True)
print("Evidências:", bundle, flush=True)
if returncode:
    print(log.read_text(encoding="utf-8", errors="replace")[-8000:])
    raise RuntimeError(f"Treino terminou com {report['status']}.")